# RAG Pipeline Walkthrough (Step by Step)

This is a **standalone notebook** that demonstrates each stage of the RAG pipeline on:

`data/uploads/Employee-Handbook.pdf`

It does not import project backend modules (`backend/*`).

Stages covered:
1. Environment and setup
2. PDF loading
3. Chunking
4. Embeddings
5. FAISS indexing
6. Retrieval with similarity scores
7. Prompt building
8. Final comparison (without RAG vs with RAG)

In [1]:
import os
from pprint import pprint

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Notebook-local configuration (independent from backend/config.py)
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
TOP_K = 4
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in .env"
print("Environment loaded successfully.")
print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}, TOP_K={TOP_K}")

Environment loaded successfully.
CHUNK_SIZE=300, CHUNK_OVERLAP=50, TOP_K=4


In [2]:
# STEP 1: Define file paths
PDF_PATH = "data/uploads/Employee-Handbook.pdf"
NB_INDEX_PATH = "storage/faiss_index_notebook"

print("PDF_PATH:", PDF_PATH)
print("Notebook index path:", NB_INDEX_PATH)
print("PDF exists:", os.path.exists(PDF_PATH))

PDF_PATH: data/uploads/Employee-Handbook.pdf
Notebook index path: storage/faiss_index_notebook
PDF exists: True


In [4]:
# STEP 2: Load PDF
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print("Total pages loaded:", len(documents))
print("First page metadata:")
pprint(documents[0].metadata)
print("\nFirst page preview:\n")
print(documents[0].page_content[:800])

Total pages loaded: 34
First page metadata:
{'creationdate': '2019-01-25T16:37:28+00:00',
 'creator': 'Microsoft Word',
 'moddate': '2019-01-25T16:37:28+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'PyPDF',
 'source': 'data/uploads/Employee-Handbook.pdf',
 'total_pages': 34}

First page preview:

Employee Handbook 
Welcome 4 
Getting to know our company 4 
Employment basics 5 
Employment contract types 5 
Equal opportunity employment 5 
Recruitment and selection process 6 
Background checks 6 
Referrals 7 
Attendance 8 
Workplace policies 8 
Confidentiality and data protection 8 
Harassment and violence 9 
Workplace harassment 10 
Workplace violence 10 
Workplace safety and health 11 
Preventative action 12 
Emergency management 12 
Smoking 12 
Drug-free workplace 13 
Employee Code of Conduct 14 
Dress code 14 
Cyber security and digital devices 14 
Internet usage 15 
Cell phone 15 
Corporate email 16 
Social media 16 
Conflict of interest 17 
Employee relationships 18 
Fratern

In [5]:
# STEP 3: Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\nChunk[0] metadata:")
pprint(chunks[0].metadata)
print("\nChunk[0] preview:\n")
print(chunks[0].page_content[:600])

Total chunks: 242

Chunk[0] metadata:
{'creationdate': '2019-01-25T16:37:28+00:00',
 'creator': 'Microsoft Word',
 'moddate': '2019-01-25T16:37:28+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'PyPDF',
 'source': 'data/uploads/Employee-Handbook.pdf',
 'total_pages': 34}

Chunk[0] preview:

Employee Handbook 
Welcome 4 
Getting to know our company 4 
Employment basics 5 
Employment contract types 5 
Equal opportunity employment 5 
Recruitment and selection process 6 
Background checks 6 
Referrals 7 
Attendance 8 
Workplace policies 8 
Confidentiality and data protection 8


In [6]:
# STEP 4: Build embeddings + FAISS index
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(NB_INDEX_PATH)

print("FAISS index built and saved.")
print("Saved files in:", NB_INDEX_PATH)
print(os.listdir(NB_INDEX_PATH))

FAISS index built and saved.
Saved files in: storage/faiss_index_notebook
['index.faiss', 'index.pkl']


In [7]:
# STEP 5: Retrieval with similarity score (ranking demo)
question = "What is the attendance policy?"
docs_and_scores = vector_store.similarity_search_with_score(question, k=TOP_K)

print("Question:", question)
print("Retrieved items:", len(docs_and_scores))

for i, (doc, score) in enumerate(docs_and_scores, start=1):
    source = os.path.basename(doc.metadata.get("source", ""))
    page = doc.metadata.get("page", 0) + 1
    print("\n" + "=" * 80)
    print(f"Rank #{i} | Similarity Score: {float(score):.6f} | Source: {source} | Page: {page}")
    print("-" * 80)
    print(doc.page_content[:500])

Question: What is the attendance policy?
Retrieved items: 4

Rank #1 | Similarity Score: 0.979038 | Source: Employee-Handbook.pdf | Page: 6
--------------------------------------------------------------------------------
program manager for more information. 
Attendance 
We expect you to be present during your scheduled working hours. If you face an 
emergency that prevents you from coming to work one day, contact your manager as 
soon as possible. We will excuse unreported absences in cases of [serious accidents,

Rank #2 | Similarity Score: 1.161355 | Source: Employee-Handbook.pdf | Page: 34
--------------------------------------------------------------------------------
to following our policies. If you need any clarifications, feel free to ask HR. 
Date: .../.../... 
………………………...

Rank #3 | Similarity Score: 1.216393 | Source: Employee-Handbook.pdf | Page: 8
--------------------------------------------------------------------------------
We may also discipline any unintentional bre

In [8]:
# STEP 6: Build grounded prompt from retrieved chunks
retrieved_docs = [doc for doc, _ in docs_and_scores]

context = "\n\n".join(
    [
        f"Source: {d.metadata.get('source')} | Page: {d.metadata.get('page')}\n{d.page_content}"
        for d in retrieved_docs
    ]
)

rag_prompt = f"""You are a document-grounded assistant.
Answer the question using only the context below.
If the answer is not available in the context, say:
"I don't know based on the provided PDF."

Context:
{context}

Question:
{question}

Answer:"""

print("Prompt preview (first 1200 chars):\n")
print(rag_prompt[:1200])

Prompt preview (first 1200 chars):

You are a document-grounded assistant.
Answer the question using only the context below.
If the answer is not available in the context, say:
"I don't know based on the provided PDF."

Context:
Source: data/uploads/Employee-Handbook.pdf | Page: 5
program manager for more information. 
Attendance 
We expect you to be present during your scheduled working hours. If you face an 
emergency that prevents you from coming to work one day, contact your manager as 
soon as possible. We will excuse unreported absences in cases of [serious accidents,

Source: data/uploads/Employee-Handbook.pdf | Page: 33
to following our policies. If you need any clarifications, feel free to ask HR. 
Date: .../.../... 
………………………...

Source: data/uploads/Employee-Handbook.pdf | Page: 7
We may also discipline any unintentional breach of this policy depending on its 
frequency and seriousness. We will terminate employees who repeatedly disregard this 
policy, even when they do so u

In [9]:
# STEP 7: Ask LLM without RAG vs with RAG
llm = ChatOpenAI(model=LLM_MODEL, temperature=TEMPERATURE)

no_rag_answer = llm.invoke(question).content
rag_answer = llm.invoke(rag_prompt).content

print("=" * 80)
print("WITHOUT RAG")
print("=" * 80)
print(no_rag_answer)

print("\n" + "=" * 80)
print("WITH RAG")
print("=" * 80)
print(rag_answer)

WITHOUT RAG
Could you please specify which organization's or institution's attendance policy you are referring to? This will help me provide the most accurate information.

WITH RAG
The attendance policy expects employees to be present during their scheduled working hours. If an emergency prevents an employee from coming to work for a day, they must contact their manager as soon as possible. Unreported absences will be excused only in cases of serious accidents.


## Notes for Teaching

- The retriever ranks chunks by semantic similarity.
- Similarity score helps explain why some chunks are selected first.
- The LLM does not search the full PDF directly.
- FAISS retrieves relevant chunks, then the LLM writes the final answer.